In [ ]:
import pandas as pd
import glob
import os

# Let's search for these files dynamically in the current directory or nearby directories
file_names = [
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_4sxH8Dz0hr.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_9xtqAsPitG.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_58yzyiBQ9U.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_bRQ9afwSIP.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_hMxhx9QLTq.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_i3BJEnT77K.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_-nwub4cROX.csv",
    "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_WdjZl0bRfZ.csv"
]

# Check possible locations
possible_dirs = [".", "..", "data", "/mnt/data"]
paths = []

for name in file_names:
    found = False
    for d in possible_dirs:
        test_path = os.path.join(d, name)
        if os.path.exists(test_path):
            paths.append(test_path)
            found = True
            break
    if not found:
        # If not found, fall back to current directory to let it raise or handle gracefully
        paths.append(name)

for p in paths:
    if os.path.exists(p):
        print(f"Found: {os.path.basename(p)} ({os.path.getsize(p)} bytes)")
        df = pd.read_csv(p, sep=";", quotechar='"', encoding="utf-8")
        # Use case-insensitive column matching or fallback to existing columns
        cols = [c for c in ['title', 'body', 'Title', 'Body', 'Content'] if c in df.columns]
        print(df.shape, list(df.columns)[:8], df[cols].head(1).to_dict('records') if cols else "No matched columns")
    else:
        print(f"Could not find: {os.path.basename(p)} in any of the checked locations.")

Found: Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_4sxH8Dz0hr.csv (7850 bytes)
(2, 34) ['original_id', 'source_name', 'tier', 'title', 'url', 'body', 'date_published', 'language'] [{'title': "'大众银行60周年庆奖不停 51辆Proton e.MAS 7及3辆BMW i5待送出！ - 国内 - 即时国内 - 市场情报", 'body': "'大众银行60周年庆奖不停 51辆Proton e.MAS 7及3辆BMW i5待送出！ 大众银行60周年“钻禧庆典”（Diamond Jubilee Campaign）持续派发大奖，第二季奖项颁发仪式日前圆满举行，来自吉隆坡、雪兰莪、槟城及沙巴的6名幸运儿，各赢取了一辆Proton e.MAS 7电动车！另有120名得主则共享总值19万2000令吉的季度现金奖。 第二季奖项颁发仪式日前在大众银行大厦举行。随着这一轮奖项揭晓，活动接下来数月仍有更多丰富奖项待送出，包括51辆Proton e.MAS 7及总值38万4000令吉季度保证现金奖。终极大奖则是3辆BMW i5！ 截至目前，活动期间已陆续送出超过400万令吉的“保证现金奖励”（Guaranteed Cash Rewards）。 此外，“钻石抽奖次数”（Diamond Entries）排名前60名的客户，也可获得最高6000令吉的保证现金奖励。 想参与抽奖，客户只需注册或购买符合资格的产品或服务，并达到指定条件，即可累积“钻石抽奖次数”；累积次数越多，中奖机会越高。 大众银行及大众伊斯兰银行的新客户更可享额外10次“钻石抽奖次数”，惟须符合相关最低条件。活动从2026年1月1日进行至12月31日。 除了通过抽奖活动回馈客户，大众银行也通过一系列60周年庆祝活动，以不同形式回馈并邀请客户及公众参与，共同见证60周年重要里程碑。 大众银行于1966年在马来西亚创立，60周年庆典以“成长与信任”（Growth and Trust）为主题，体现客户长期以来的信任与支持，是该银行走过60年历程的重要基础。60周年不仅是回顾过去，也标志着下一阶段发展的新起点。 更多详情可浏览大众银

In [ ]:
import json
summary=[]
dfs=[]
for p in paths:
    d=pd.read_csv(p, sep=";", quotechar='"', encoding="utf-8")
    d["source_file"]=os.path.basename(p)
    # normalize leading apostrophe only on values where export artifact
    for col in d.select_dtypes(include="object").columns:
        d[col]=d[col].map(lambda x: x[1:] if isinstance(x,str) and x.startswith("'") else x)
    dfs.append(d)
    summary.append({"file":os.path.basename(p),"rows":len(d),"cols":len(d.columns),"entities_nonempty":int(d["entities"].notna().sum()) if "entities" in d.columns else None,"language_counts":d["language"].value_counts(dropna=False).to_dict() if "language" in d.columns else {}})
all_df=pd.concat(dfs,ignore_index=True,sort=False)
print("FILE SUMMARY")
print(json.dumps(summary,ensure_ascii=False,indent=2)[:10000])
print("total rows",len(all_df),"cols",len(all_df.columns))
print("columns",list(all_df.columns))
print("id unique",all_df["original_id"].nunique(),"missing ids",all_df["original_id"].isna().sum())
print("dup IDs",all_df["original_id"].duplicated().sum())
print("entities nonempty",all_df["entities"].notna().sum() if "entities" in all_df else "no entities col")
print("languages",all_df["language"].value_counts(dropna=False).to_dict())
print("title missing",all_df["title"].isna().sum(),"body missing",all_df["body"].isna().sum())

FILE SUMMARY
[
  {
    "file": "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_4sxH8Dz0hr.csv",
    "rows": 2,
    "cols": 35,
    "entities_nonempty": 0,
    "language_counts": {
      "zh": 1,
      "ms": 1
    }
  },
  {
    "file": "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_9xtqAsPitG.csv",
    "rows": 96,
    "cols": 35,
    "entities_nonempty": 0,
    "language_counts": {
      "en": 79,
      "ms": 14,
      "zh": 3
    }
  },
  {
    "file": "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_58yzyiBQ9U.csv",
    "rows": 105,
    "cols": 35,
    "entities_nonempty": 0,
    "language_counts": {
      "en": 97,
      "ms": 6,
      "zh": 2
    }
  },
  {
    "file": "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_bRQ9afwSIP.csv",
    "rows": 347,
    "cols": 35,
    "entities_nonempty": 0,
    "language_counts": {
      "en": 289,
      "ms": 48,
      "id": 7,
      "zh": 3
    }
  },
  {
    "file": "Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_

In [ ]:
# check conflict IDs
id_check = all_df.groupby("original_id", dropna=False).agg(title_versions=("title","nunique"), body_versions=("body","nunique"))
conflicts = id_check[(id_check.title_versions>1)|(id_check.body_versions>1)]
clean = all_df.drop_duplicates(subset=["original_id"], keep="first").copy()
english = clean[clean["language"].astype(str).str.lower().str.strip().isin(["en","eng","english"])].copy()
print(len(clean), len(english), len(conflicts))
print("clipping counts", english["clipping"].value_counts(dropna=False).head(20).to_dict() if "clipping" in english else None)
print("article fields", english)

In [ ]:
import pandas as pd, os, json, zipfile, random
from pathlib import Path

file_names = [
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_4sxH8Dz0hr.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_9xtqAsPitG.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_58yzyiBQ9U.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_bRQ9afwSIP.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_hMxhx9QLTq.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_i3BJEnT77K.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_-nwub4cROX.csv",
"Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_WdjZl0bRfZ.csv",
]

# Dynamically find where the files are located in the environment
possible_dirs = [".", "/content", "/mnt/data", "data"]
paths = []
for name in file_names:
    found = False
    for d in possible_dirs:
        test_path = os.path.join(d, name)
        if os.path.exists(test_path):
            paths.append(test_path)
            found = True
            break
    if not found:
        paths.append(name)

dfs=[]
for p in paths:
    d=pd.read_csv(p, sep=";", quotechar='"', encoding="utf-8")
    d["source_file"] = os.path.basename(p)
    for col in d.select_dtypes(include="object").columns:
        d[col]=d[col].map(lambda x: x[1:] if isinstance(x,str) and x.startswith("'") else x)
    dfs.append(d)
all_df=pd.concat(dfs, ignore_index=True, sort=False)
id_check=all_df.groupby("original_id", dropna=False).agg(title_versions=("title","nunique"), body_versions=("body","nunique"))
conflicts=id_check[(id_check.title_versions>1)|(id_check.body_versions>1)]
clean=all_df.drop_duplicates(subset=["original_id"], keep="first").copy()
english=clean[clean["language"].astype(str).str.lower().str.strip().isin(["en","eng","english"])].copy()
# stratified sample across clipping categories, then assign split using deterministic random order
random_state=42
queue=english.groupby("clipping", group_keys=False, dropna=False).apply(
    lambda g: g.sample(n=min(len(g), max(1, round(120*len(g)/len(english)))), random_state=random_state)
).reset_index(drop=True)
# adjust to exactly 120 if group rounding differs
if len(queue)>120:
    queue=queue.sample(n=120, random_state=random_state)
elif len(queue)<120:
    remaining=english[~english["original_id"].isin(queue["original_id"])]
    queue=pd.concat([queue, remaining.sample(n=120-len(queue), random_state=random_state)], ignore_index=True)
queue=queue.sample(frac=1, random_state=random_state).reset_index(drop=True)
queue["split"] = ["train"]*84 + ["validation"]*18 + ["test"]*18
# use shorter and stable columns for annotation
annot_cols = [c for c in ["original_id","title","body","clipping","source_name","date_published","split"] if c in queue.columns]
annotation_queue=queue[annot_cols].copy()
annotation_queue["entities_json"] = ""
profile = pd.DataFrame([
    ["Jumlah CSV", 8],
    ["Total baris sebelum deduplikasi", len(all_df)],
    ["Jumlah original_id unik", int(all_df["original_id"].nunique())],
    ["ID dengan judul/isi berbeda", len(conflicts)],
    ["Jumlah setelah deduplikasi original_id", len(clean)],
    ["Artikel bahasa Inggris setelah deduplikasi", len(english)],
    ["Baris dengan entities terisi", int(all_df["entities"].notna().sum()) if "entities" in all_df.columns else "kolom tidak ditemukan"],
    ["Jumlah artikel dalam antrean anotasi", len(annotation_queue)],
    ["Train / validation / test dalam antrean", "84 / 18 / 18"],
], columns=["Pemeriksaan","Hasil"])
lang = all_df["language"].value_counts(dropna=False).rename_axis("language").reset_index(name="rows")
guide = pd.DataFrame([
["Tujuan","Tandai setiap entitas bernama yang benar-benar muncul pada teks artikel."],
["PERSON","Nama orang, misalnya Anwar Ibrahim atau Thong Pak Leng."],
["ORG","Organisasi, perusahaan, bank, partai, lembaga, misalnya Maybank atau Bursa Malaysia."],
["GPE","Negara, kota, wilayah politik/geografis yang berfungsi sebagai entitas geopolitik, misalnya Malaysia atau Kuala Lumpur."],
["LOC","Lokasi fisik yang bukan GPE, misalnya Selat Melaka atau Gunung Kinabalu."],
["DATE","Tanggal, waktu, atau rentang waktu eksplisit, misalnya 7 October 2026."],
["EVENT","Nama acara/peristiwa tertentu, misalnya Formula 1 Gulf Air Bahrain Grand Prix."],
["Span","Tandai teks persis seperti yang muncul; jangan menambahkan kata yang tidak ada."],
["Offset","start_char adalah indeks karakter awal (inklusif), end_char indeks akhir (eksklusif), dihitung pada kolom title atau body secara terpisah."],
["Format JSON","Isi entities_json dengan array objek seperti contoh di bawah; gunakan field title/body."],
["Contoh JSON",'[{"text":"Maybank","label":"ORG","field":"title","start_char":0,"end_char":7}]'],
["Konsistensi","Jangan melabeli kata umum sebagai entitas. Jika ragu, diskusikan dengan anggota lain dan tulis keputusan di catatan."],
["Split","Label train untuk melatih, validation untuk memilih model, test untuk evaluasi final. Jangan memakai label test untuk melatih atau mengubah model."],
["Peringatan","Antrean 120 artikel adalah sampel awal. Periksa bahwa tiap artikel sesuai cakupan studi kasus sebelum anotasi final."],
], columns=["Bagian","Panduan"])

# Use local directories for writing outputs to avoid missing mount issues
outdir="./ner_dataset_prepared"
os.makedirs(outdir, exist_ok=True)
csv_clean=os.path.join(outdir,"all_articles_deduplicated.csv")
csv_eng=os.path.join(outdir,"english_articles_deduplicated.csv")
csv_queue=os.path.join(outdir,"annotation_queue_120.csv")
xlsx=os.path.join(outdir,"NER_annotation_workbook.xlsx")
clean.to_csv(csv_clean,index=False,encoding="utf-8-sig")
english.to_csv(csv_eng,index=False,encoding="utf-8-sig")
annotation_queue.to_csv(csv_queue,index=False,encoding="utf-8-sig")
with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    profile.to_excel(writer,index=False,sheet_name="Data_Profile")
    lang.to_excel(writer,index=False,sheet_name="Language_Counts")
    english[annot_cols[:-1] if False else [c for c in ["original_id","title","body","clipping","source_name","date_published"] if c in english.columns]].to_excel(writer,index=False,sheet_name="English_656")
    annotation_queue.to_excel(writer,index=False,sheet_name="Annotation_Queue")
    guide.to_excel(writer,index=False,sheet_name="Annotation_Guide")
    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for col_cells in ws.columns:
            letter=col_cells[0].column_letter
            max_len=max((len(str(c.value)) if c.value is not None else 0) for c in col_cells[:50])
            ws.column_dimensions[letter].width=min(max(max_len+2,12),45)
# zip cleaned csv outputs + workbook
zip_path="./Maybank_NER_prepared_data.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in [csv_clean,csv_eng,csv_queue,xlsx]:
        z.write(p, arcname=os.path.basename(p))
print("Files created:", xlsx, zip_path, csv_eng, csv_queue)

/tmp/ipykernel_2709/1408551114.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  queue=english.groupby("clipping", group_keys=False, dropna=False).apply(


Files created: ./ner_dataset_prepared/NER_annotation_workbook.xlsx ./Maybank_NER_prepared_data.zip ./ner_dataset_prepared/english_articles_deduplicated.csv ./ner_dataset_prepared/annotation_queue_120.csv


In [ ]:
from google.colab import files

files.download(
    "./ner_dataset_prepared/NER_annotation_workbook.xlsx"
)

In [ ]:
import nbformat as nbf, os, json, textwrap
from pathlib import Path

# Correcting path to local directory to prevent FileNotFoundError
out = "./NER_training_evaluation_next_steps.ipynb"
nb = nbf.v4.new_notebook()
cells = []

cells.append(nbf.v4.new_markdown_cell("""# Tahap 2 — Training dan Evaluasi Model NER

Notebook ini digunakan **setelah** sheet `Annotation_Queue` pada `NER_annotation_workbook.xlsx` selesai diisi dan diperiksa.

**Penting:**
- Jangan menjalankan training sebelum anotasi selesai.
- `entities_json` harus berisi JSON valid sesuai `Annotation_Guide`.
- Label yang dipakai: `PERSON`, `ORG`, `GPE`, `LOC`, `DATE`, `EVENT`.
- `start_char` inklusif dan `end_char` eksklusif; `field` harus `title` atau `body`.
- Data `test` hanya untuk evaluasi akhir, bukan untuk training.
- Karena jumlah data berlabel kecil, hasil awal bisa rendah dan belum tentu mewakili kinerja pada semua artikel.
"""))

cells.append(nbf.v4.new_code_cell("""!pip -q install "spacy>=3.7,<4" openpyxl
"""))

cells.append(nbf.v4.new_markdown_cell("""## 1. Unggah workbook anotasi

Unggah file Excel yang sudah kalian isi. Jika namanya berbeda, kode di bawah akan menggunakan file yang dipilih."""))

cells.append(nbf.v4.new_code_cell("""from google.colab import files
uploaded = files.upload()

xlsx_path = next((name for name in uploaded if name.lower().endswith(".xlsx")), None)
if xlsx_path is None:
    raise ValueError("Unggah workbook .xlsx yang berisi sheet Annotation_Queue.")

print("Workbook:", xlsx_path)
"""))

cells.append(nbf.v4.new_code_cell("""import pandas as pd
import json, math, os, random, re
from collections import Counter, defaultdict

queue = pd.read_excel(xlsx_path, sheet_name="Annotation_Queue")
required_cols = {"title", "body", "split", "entities_json"}
missing = required_cols - set(queue.columns)
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

queue["entities_json"] = queue["entities_json"].fillna("").astype(str).str.strip()
print("Jumlah artikel dalam antrean:", len(queue))
print("Jumlah anotasi terisi:", (queue["entities_json"] != "").sum())
print("Distribusi split:", queue["split"].value_counts(dropna=False).to_dict())
"""))

cells.append(nbf.v4.new_markdown_cell("""## 2. Validasi anotasi

Cell ini memeriksa JSON, label, nama field, offset, kecocokan teks, dan entitas yang saling tumpang tindih. Perbaiki semua error sebelum melanjutkan."""))

cells.append(nbf.v4.new_code_cell("""LABELS = {"PERSON", "ORG", "GPE", "LOC", "DATE", "EVENT"}
FIELDS = {"title", "body"}

def parse_entities(value):
    if not value:
        return []
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError("entities_json harus berupa array JSON")
    return parsed

errors = []
parsed_rows = []
for idx, row in queue.iterrows():
    raw = row["entities_json"]
    try:
        ents = parse_entities(raw)
    except Exception as e:
        errors.append((idx, str(row.get("original_id", "")), f"JSON tidak valid: {e}"))
        parsed_rows.append([])
        continue

    valid_ents = []
    by_field = defaultdict(list)
    for j, ent in enumerate(ents):
        try:
            label = ent["label"]
            field = ent["field"]
            start = int(ent["start_char"])
            end = int(ent["end_char"])
            txt = ent["text"]
            if label not in LABELS:
                errors.append((idx, str(row.get("original_id", "")), f"Label tidak dikenal: {label}"))
            if field not in FIELDS:
                errors.append((idx, str(row.get("original_id", "")), f"Field tidak valid: {field}"))
                continue
            source_text = str(row[field]) if pd.notna(row[field]) else ""
            if not (0 <= start < end <= len(source_text)):
                errors.append((idx, str(row.get("original_id", "")), f"Offset di luar teks untuk {txt!r}"))
                continue
            if source_text[start:end] != txt:
                errors.append((idx, str(row.get("original_id", "")), f"Mismatch: tersimpan {txt!r}, teks offset menghasilkan {source_text[start:end]!r}"))
            by_field[field].append((start, end, label, txt))
            valid_ents.append(ent)
        except Exception as e:
            errors.append((idx, str(row.get("original_id", "")), f"Entitas #{j} tidak lengkap: {e}"))

    for field, spans in by_field.items():
        spans = sorted(spans)
        for a, b in zip(spans, spans[1:]):
            if b[0] < a[1]:
                errors.append((idx, str(row.get("original_id", "")), f"Entitas overlap di {field}: {a[3]!r} dan {b[3]!r}"))
    parsed_rows.append(valid_ents)

queue["_parsed_entities"] = parsed_rows
print("Jumlah error validasi:", len(errors))
for err in errors[:50]:
    print(err)
if errors:
    raise ValueError("Hentikan proses: perbaiki error di atas dalam workbook, unggah ulang, lalu jalankan ulang dari awal.")
"""))

cells.append(nbf.v4.new_markdown_cell("""## 3. Siapkan contoh training dari title dan body

Title dan body diperlakukan sebagai teks terpisah karena offset anotasi dihitung secara terpisah. Baris tanpa entitas tetap dapat dipakai sebagai contoh negatif hanya jika kelompok sudah memeriksa bahwa memang tidak ada entitas sesuai skema."""))

cells.append(nbf.v4.new_code_cell("""from spacy.training import Example
import spacy
from spacy.util import minibatch, compounding

def make_examples(frame, nlp, require_annotations=True):
    examples = []
    skipped = 0
    for _, row in frame.iterrows():
        ents = row["_parsed_entities"]
        # Empty JSON is ambiguous: it may mean no entities or not annotated.
        # We require each included article to have an explicit JSON array, e.g. [] if no entities.
        if not str(row["entities_json"]).strip():
            skipped += 1
            continue
        for field in ("title", "body"):
            text = "" if pd.isna(row[field]) else str(row[field])
            spans = []
            for ent in ents:
                if ent["field"] == field:
                    spans.append((int(ent["start_char"]), int(ent["end_char"]), ent["label"]))
            doc = nlp.make_doc(text)
            examples.append(Example.from_dict(doc, {"entities": spans}))
    return examples, skipped

annotated = queue[queue["entities_json"].str.len() > 0].copy()
if annotated.empty:
    raise ValueError("Belum ada anotasi. Isi entities_json terlebih dahulu; gunakan [] secara eksplisit bila artikel telah diperiksa dan tidak memiliki entitas.")

print("Artikel yang dianotasi:", len(annotated))
print("Entitas per label:", Counter(e["label"] for ents in annotated["_parsed_entities"] for e in ents))
print("Split anotasi:", annotated["split"].value_counts().to_dict())
"""))

cells.append(nbf.v4.new_markdown_cell("""## 4. Latih model NER

Model ini dilatih dari pipeline spaCy bahasa Inggris yang kosong, menggunakan anotasi kelompok. Ini adalah baseline yang bisa diulang; karena dataset anotasi relatif kecil, jangan mengklaim hasilnya tinggi sebelum melihat evaluasi."""))

cells.append(nbf.v4.new_code_cell("""random.seed(42)
spacy.util.fix_random_seed(42)

train_df = annotated[annotated["split"].astype(str).str.lower().eq("train")].copy()
val_df = annotated[annotated["split"].astype(str).str.lower().isin(["validation", "val", "dev"])].copy()
test_df = annotated[annotated["split"].astype(str).str.lower().eq("test")].copy()

if train_df.empty:
    raise ValueError("Tidak ada artikel beranotasi dengan split='train'.")
if test_df.empty:
    raise ValueError("Tidak ada artikel beranotasi dengan split='test'.")
if not train_df["_parsed_entities"].map(len).sum():
    raise ValueError("Data train tidak memiliki entitas berlabel. Periksa anotasi dan label split.")

nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")
for label in sorted(LABELS):
    ner.add_label(label)

train_examples, _ = make_examples(train_df, nlp)
val_examples, _ = make_examples(val_df, nlp) if not val_df.empty else ([], 0)

optimizer = nlp.initialize(lambda: train_examples)
N_EPOCHS = 20
history = []

for epoch in range(N_EPOCHS):
    random.shuffle(train_examples)
    losses = {}
    batches = minibatch(train_examples, size=compounding(4.0, 16.0, 1.5))
    for batch in batches:
        nlp.update(batch, sgd=optimizer, drop=0.2, losses=losses)
    history.append({"epoch": epoch + 1, "ner_loss": losses.get("ner", 0.0)})
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{N_EPOCHS} — loss: {losses.get(\'ner\', 0.0):.3f}")

pd.DataFrame(history)
"""))

cells.append(nbf.v4.new_markdown_cell("""## 5. Evaluasi exact-span pada data test

Sebuah entitas dianggap benar jika **label, start, dan end** cocok persis. Nilai berikut dihitung dari prediksi model pada test set; jangan mengganti atau menebak hasilnya."""))

cells.append(nbf.v4.new_code_cell("""def gold_spans_for(row, field):
    return {
        (int(e["start_char"]), int(e["end_char"]), e["label"])
        for e in row["_parsed_entities"] if e["field"] == field
    }

def evaluate_exact_span(nlp, frame):
    counts = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    total = {"tp": 0, "fp": 0, "fn": 0}
    details = []

    for _, row in frame.iterrows():
        if not str(row["entities_json"]).strip():
            continue
        for field in ("title", "body"):
            text = "" if pd.isna(row[field]) else str(row[field])
            doc = nlp(text)
            gold = gold_spans_for(row, field)
            pred = {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents}
            for item in gold & pred:
                counts[item[2]]["tp"] += 1
                total["tp"] += 1
            for item in pred - gold:
                counts[item[2]]["fp"] += 1
                total["fp"] += 1
            for item in gold - pred:
                counts[item[2]]["fn"] += 1
                total["fn"] += 1
            if gold != pred:
                details.append({
                    "original_id": row.get("original_id", ""),
                    "field": field,
                    "text": text[:250],
                    "gold": sorted(gold),
                    "pred": sorted(pred)
                })

    rows = []
    for label in sorted(LABELS):
        tp, fp, fn = counts[label]["tp"], counts[label]["fp"], counts[label]["fn"]
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"label": label, "precision": precision, "recall": recall, "f1": f1,
                     "TP": tp, "FP": fp, "FN": fn})
    tp, fp, fn = total["tp"], total["fp"], total["fn"]
    micro_p = tp / (tp + fp) if tp + fp else 0.0
    micro_r = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0
    return pd.DataFrame(rows), {"micro_precision": micro_p, "micro_recall": micro_r, "micro_f1": micro_f1,
                                "TP": tp, "FP": fp, "FN": fn}, pd.DataFrame(details)

metrics_by_label, micro_metrics, error_examples = evaluate_exact_span(nlp, test_df)
display(metrics_by_label.round(4))
print("Micro exact-span metrics:", micro_metrics)
print("Jumlah field title/body yang memiliki perbedaan gold vs prediksi:", len(error_examples))
display(error_examples.head(10))
"""))

cells.append(nbf.v4.new_markdown_cell("""## 6. Simpan model dan hasil evaluasi

Simpan model, metrik, dan contoh error agar dapat dimasukkan ke laporan. Pastikan hasil metrik di laporan sama dengan hasil yang keluar saat notebook dijalankan."""))

cells.append(nbf.v4.new_code_cell("""model_dir = "/content/ner_model"
nlp.to_disk(model_dir)
metrics_by_label.to_csv("/content/ner_metrics_by_label.csv", index=False)
error_examples.to_json("/content/ner_error_examples.json", orient="records", force_ascii=False, indent=2)

print("Model disimpan di:", model_dir)
print("Metrik disimpan di: /content/ner_metrics_by_label.csv")
print("Contoh error disimpan di: /content/ner_error_examples.json")
"""))

cells.append(nbf.v4.new_code_cell("""# Uji prediksi pada teks baru (contoh demonstrasi, bukan evaluasi)
demo = "Anwar Ibrahim met representatives of Maybank in Kuala Lumpur on 7 October 2026."
doc = nlp(demo)
print([(ent.text, ent.label_, ent.start_char, ent.end_char) for ent in doc.ents])
"""))

nb["cells"] = cells
nb["metadata"] = {
    "colab": {"name": "NER_training_evaluation_next_steps.ipynb", "provenance": []},
    "kernelspec": {"display_name": "Python 3", "name": "python3"},
    "language_info": {"name": "python"}
}
nbf.write(nb, out)
print(out, os.path.getsize(out))

./NER_training_evaluation_next_steps.ipynb 16135


In [ ]:
!pip -q install "spacy>=3.7,<4" openpyxl

In [ ]:
from google.colab import files
uploaded = files.upload()

xlsx_path = next((name for name in uploaded if name.lower().endswith(".xlsx")), None)
if xlsx_path is None:
    raise ValueError("Unggah workbook .xlsx yang berisi sheet Annotation_Queue.")

print("Workbook:", xlsx_path)

Saving NER_annotation_workbook (1).xlsx to NER_annotation_workbook (1).xlsx
Workbook: NER_annotation_workbook (1).xlsx


In [ ]:
import pandas as pd
import json, math, os, random, re
from collections import Counter, defaultdict

queue = pd.read_excel(xlsx_path, sheet_name="Annotation_Queue")
required_cols = {"title", "body", "split", "entities_json"}
missing = required_cols - set(queue.columns)
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

queue["entities_json"] = queue["entities_json"].fillna("").astype(str).str.strip()
print("Jumlah artikel dalam antrean:", len(queue))
print("Jumlah anotasi terisi:", (queue["entities_json"] != "").sum())
print("Distribusi split:", queue["split"].value_counts(dropna=False).to_dict())

Jumlah artikel dalam antrean: 120
Jumlah anotasi terisi: 0
Distribusi split: {'train': 84, 'validation': 18, 'test': 18}


In [ ]:
LABELS = {"PERSON", "ORG", "GPE", "LOC", "DATE", "EVENT"}
FIELDS = {"title", "body"}

def parse_entities(value):
    if not value:
        return []
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError("entities_json harus berupa array JSON")
    return parsed

errors = []
parsed_rows = []
for idx, row in queue.iterrows():
    raw = row["entities_json"]
    try:
        ents = parse_entities(raw)
    except Exception as e:
        errors.append((idx, str(row.get("original_id", "")), f"JSON tidak valid: {e}"))
        parsed_rows.append([])
        continue

    valid_ents = []
    by_field = defaultdict(list)
    for j, ent in enumerate(ents):
        try:
            label = ent["label"]
            field = ent["field"]
            start = int(ent["start_char"])
            end = int(ent["end_char"])
            txt = ent["text"]
            if label not in LABELS:
                errors.append((idx, str(row.get("original_id", "")), f"Label tidak dikenal: {label}"))
            if field not in FIELDS:
                errors.append((idx, str(row.get("original_id", "")), f"Field tidak valid: {field}"))
                continue
            source_text = str(row[field]) if pd.notna(row[field]) else ""
            if not (0 <= start < end <= len(source_text)):
                errors.append((idx, str(row.get("original_id", "")), f"Offset di luar teks untuk {txt!r}"))
                continue
            if source_text[start:end] != txt:
                errors.append((idx, str(row.get("original_id", "")), f"Mismatch: tersimpan {txt!r}, teks offset menghasilkan {source_text[start:end]!r}"))
            by_field[field].append((start, end, label, txt))
            valid_ents.append(ent)
        except Exception as e:
            errors.append((idx, str(row.get("original_id", "")), f"Entitas #{j} tidak lengkap: {e}"))

    for field, spans in by_field.items():
        spans = sorted(spans)
        for a, b in zip(spans, spans[1:]):
            if b[0] < a[1]:
                errors.append((idx, str(row.get("original_id", "")), f"Entitas overlap di {field}: {a[3]!r} dan {b[3]!r}"))
    parsed_rows.append(valid_ents)

queue["_parsed_entities"] = parsed_rows
print("Jumlah error validasi:", len(errors))
for err in errors[:50]:
    print(err)
if errors:
    raise ValueError("Hentikan proses: perbaiki error di atas dalam workbook, unggah ulang, lalu jalankan ulang dari awal.")

Jumlah error validasi: 0


In [ ]:
import pandas as pd, os, json, openpyxl, re
path='/mnt/data/ner_dataset_prepared/NER_annotation_queue_draft_filled'
os.path.exists(path)

False

In [ ]:
import os, pandas as pd, json, openpyxl, re
path='/mnt/data/ner_dataset_prepared/NER_annotation_queue_draft_filled'
print(os.path.exists(path))
if os.path.exists(path):
    xls=pd.ExcelFile(path)
    print(xls.sheet_names)
    df=pd.read_excel(path,sheet_name='Annotation_Queue')
    print(df.shape, df.columns.tolist())
    print(df[['original_id','title','body','split','entities_json']].head(3).to_string(index=False)[:3000])

False


In [ ]:
try:
 import spacy
 print("spacy",spacy.__version__)
 print(spacy.util.get_installed_models())
except Exception as e: print(type(e).__name__,e)

spacy 3.8.16
['en_core_web_sm']


In [ ]:
import pandas as pd
# Menggunakan file workbook yang diunggah sebelumnya
xlsx_file = "NER_annotation_workbook (1).xlsx"
df = pd.read_excel(xlsx_file, sheet_name='Annotation_Queue')
pd.set_option('display.max_colwidth', 180)
print(df[['original_id', 'title', 'body', 'split', 'entities_json']].head(8).to_string(index=False))

                                                     original_id                                                                                                                        title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
print(df['title'].head(15).tolist())

['FBM KLCI charts slight gain at midday | KLSE Screener', 'Diplomasi Sukan Anwar Letakkan Malaysia Kembali Di Persada Global – Penganalisis', 'World Bank Supports Malaysia’s Possible Integration of GST Elements Into SST', 'Bursa Malaysia ends slightly higher as technology counters dominate trading', 'Bursa Malaysia ends lower, KLCI slides 1.26pc on cautious sentiment', 'Scheduling boo-boo: Malaysian hockey league matches originally set for Deepavali rescheduled - Twentytwo13', 'Maybank Group Sustainability Week 2026 advances net zero ambitions through capacity building and collective action | Maybank', 'Stronger MyCC Powers Put Competition Compliance Higher On Governance Agenda, CIMB Report', 'PTT Synergy beli 4 kilang dalam pembinaan bernilai RM32 juta di Klang sebelum disewakan', "MGS sell-off set to hit RHB's Q3 earnings: HLIB", 'CIMB Securities lifts 2026 GDP growth forecast to 5pct on domestic strength', '马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨', 'Mesiniaga Wins Two-year RM32.3m Contract Ext

In [ ]:
# Create automatic draft annotations using conservative gazetteers and date patterns.
import re, json, pandas as pd, os
xlsx_file = "NER_annotation_workbook (1).xlsx"
df=pd.read_excel(xlsx_file, sheet_name='Annotation_Queue', dtype={'original_id':str})
months=r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
date_patterns=[
    re.compile(r'\b'+months+r'\s+\d{1,2}(?:[-–]\d{1,2})?(?:,?\s+\d{4})?\b', re.I),
    re.compile(r'\b\d{1,2}\s+'+months+r'(?:\s+\d{4})?\b', re.I),
    re.compile(r'\b(?:19|20)\d{2}\b'),
    re.compile(r'\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b', re.I),
    re.compile(r'\b(?:today|yesterday|tomorrow)\b', re.I)
]
# Curated high-confidence entities often present in these Malaysian financial news articles
orgs = [
"Maybank Group","Maybank","Bursa Malaysia","FBM KLCI","CIMB Securities","CIMB","RHB","HLIB","Hong Leong Investment Bank",
"Bank Negara Malaysia","World Bank","Malaysia Competition Commission","MyCC","Dewan Rakyat","Parliament","Tenaga Nasional Berhad",
"Tenaga Nasional","TNB Thunderbolts","Terengganu Hockey Team","MHC","PNB Merdeka Ventures","PNB","UN Global Compact Network Malaysia, Brunei & Cambodia",
"UN Global Compact Network Malaysia","UN Global Compact","Etiqa","Mesiniaga","Customs Department","Malaysian Hockey Confederation",
"Asian Games","Asian Champions Trophy","Sultan of Johor Cup","Murugappa Gold Cup","Junior World Cup","Razak Cup",
"Maybank Foundation","CIMB Report","KLSE Screener","Twentytwo13","Reuters","Bloomberg","Malaysia Airlines","Petronas","Sime Darby",
"Maybank Group Sustainability Week","MGSW","MSPC","Competition Appeal Tribunal","MyCC"
]
gpes = ["Kuala Lumpur","Malaysia","Singapore","Indonesia","Philippines","Cambodia","Brunei","Vietnam","China","India","Johor","Klang","Penang","Sabah","Sarawak","Selangor","Terengganu","Negeri Sembilan","ASEAN","Asia","United States","US","United Kingdom","Thailand","Hong Kong","Taiwan"]
# Honorific-based person names, high precision only
honorifics = re.compile(r'\b(?:Dato(?:\x27|’)?\s*(?:Sri|Seri)?|Datuk|Tan Sri|Tun|Dr\.?|Prof\.?)\s+[A-Z][A-Za-zÀ-ÿ\x27’.-]+(?:\s+[A-Z][A-Za-zÀ-ÿ\x27’.-]+){0,4}')
def find_matches(text, field):
    if not isinstance(text,str): text="" if pd.isna(text) else str(text)
    candidates=[]
    for term in sorted(orgs, key=len, reverse=True):
        for m in re.finditer(r'(?<!\w)'+re.escape(term)+r'(?!\w)', text, re.I):
            label="EVENT" if term in {"Asian Games","Asian Champions Trophy","Sultan of Johor Cup","Murugappa Gold Cup","Junior World Cup","Razak Cup","Maybank Group Sustainability Week","MGSW","MSPC"} else "ORG"
            candidates.append((m.start(),m.end(),label,m.group()))
    for term in sorted(gpes,key=len,reverse=True):
        for m in re.finditer(r'(?<!\w)'+re.escape(term)+r'(?!\w)',text,re.I):
            candidates.append((m.start(),m.end(),"GPE",m.group()))
    for pat in date_patterns:
        for m in pat.finditer(text):
            candidates.append((m.start(),m.end(),"DATE",m.group()))
    for m in honorifics.finditer(text):
        candidates.append((m.start(),m.end(),"PERSON",m.group()))
    # Deduplicate; prefer longest match, then priority PERSON/ORG/GPE/DATE/EVENT
    priority={"PERSON":0,"ORG":1,"EVENT":2,"GPE":3,"DATE":4}
    candidates.sort(key=lambda x:(x[0],-(x[1]-x[0]),priority[x[2]]))
    chosen=[]
    for cand in candidates:
        s,e,l,t=cand
        if any(not (e<=cs or s>=ce) for cs,ce,_,_ in chosen):
            continue
        chosen.append(cand)
    chosen.sort()
    return [{"text":t,"label":l,"field":field,"start_char":s,"end_char":e} for s,e,l,t in chosen]

results=[]
for i,row in df.iterrows():
    ents=[]
    for fld in ("title","body"):
        val=row.get(fld,"")
        if pd.isna(val): val=""
        ents.extend(find_matches(str(val),fld))
    # don't falsely assert [] when no reliable match; leave blank for manual review
    results.append(json.dumps(ents,ensure_ascii=False) if ents else "")
df["entities_json"]=results
print("rows with suggested annotations",sum(bool(x) for x in results),"of",len(results))
print("total suggestions",sum(len(json.loads(x)) for x in results if x))
print("labels", {lab:sum(1 for x in results if x for e in json.loads(x) if e["label"]==lab) for lab in ["PERSON","ORG","GPE","LOC","DATE","EVENT"]})

rows with suggested annotations 119 of 120
total suggestions 2376
labels {'PERSON': 52, 'ORG': 711, 'GPE': 722, 'LOC': 0, 'DATE': 848, 'EVENT': 43}


In [ ]:
from openpyxl import load_workbook
# Menggunakan file workbook yang valid di dalam sesi Colab
xlsx_file = "NER_annotation_workbook (1).xlsx"
wb = load_workbook(xlsx_file)
ws = wb["Annotation_Queue"]
headers = [c.value for c in ws[1]]
headers

['original_id',
 'title',
 'body',
 'clipping',
 'source_name',
 'date_published',
 'split',
 'entities_json']

In [ ]:
issues=[]
for _,row in df.iterrows():
    raw=row['entities_json']
    if not isinstance(raw,str) or not raw.strip(): continue
    try: ents=json.loads(raw)
    except Exception as e: issues.append((row['original_id'],'json',str(e))); continue
    for ent in ents:
        textval=str(row[ent['field']]) if not pd.isna(row[ent['field']]) else ""
        s,e=ent['start_char'],ent['end_char']
        if textval[s:e] != ent['text']:
            issues.append((row['original_id'],ent['text'],textval[s:e],s,e))
len(issues),issues[:3]

(0, [])

In [ ]:
from openpyxl import load_workbook
import os

# Menggunakan workbook yang telah diunggah di lingkungan Colab
source_path = "NER_annotation_workbook (1).xlsx"
out = "NER_annotation_workbook_draft_filled.xlsx"

wb = load_workbook(source_path)
ws = wb["Annotation_Queue"]
headers = [c.value for c in ws[1]]
json_col = headers.index("entities_json") + 1
status_col = len(headers) + 1
ws.cell(row=1, column=status_col, value="annotation_status")

for r, raw in enumerate(results, start=2):
    ws.cell(row=r, column=json_col, value=raw if raw else None)
    ws.cell(row=r, column=status_col, value="DRAFT_AUTO_REVIEW_REQUIRED" if raw else "NEEDS_MANUAL_ANNOTATION")

# Tambahkan catatan penting di sheet Annotation_Guide
guide = wb["Annotation_Guide"]
guide.cell(row=guide.max_row + 2, column=1, value="IMPORTANT: Auto-generated draft annotations")
guide.cell(row=guide.max_row, column=1, value="These annotations are machine-generated suggestions using conservative rules/gazetteers, not verified gold labels. Review every entity, label, and character offset against title/body before model training/evaluation. Blank entities_json means no reliable rule match was found; it does NOT mean the article has no entities.")

wb.save(out)
print("Saved updated workbook to:", out, f"({os.path.getsize(out)} bytes)")

Saved updated workbook to: NER_annotation_workbook_draft_filled.xlsx (1017264 bytes)


In [ ]:
import os

# Memperbaiki path agar menyimpan di direktori lokal Colab, bukan /mnt/data/
csvout = 'NER_annotation_queue_draft_filled.csv'
df.to_csv(csvout, index=False, encoding='utf-8-sig')
print(f"File berhasil disimpan ke: {csvout} ({os.path.getsize(csvout)} bytes)")

File berhasil disimpan ke: NER_annotation_queue_draft_filled.csv (706501 bytes)
